# About recursion

Get the multiplication of all numbers from 1 to a given number `n`:

In [ ]:
def sum1(n:Int) = 
  var sum = 0
  for i <- (1 to n) do
    sum = sum + i
  sum

sum1(5)

Another option is to use a recursive version of the function:

In [ ]:
def sum2(n:Int):Int =
  if (n > 1) then 
    sum2(n-1) + n
  else n

sum2(5)

In [ ]:
sum2(50000)

We can have efficiency problems as the stack gets quickly filled with thousands of calls waiting to be completed.

We can try a slightly different version with tail recursion. This means that the recursive call is the last thing to be done.

In [ ]:
def sum3(n:Int,current:Int=0):Int =
  if (n == 1) then current + n
  else sum3(n-1,current + n)

sum3(50000)

In scala the `@tailrec` annotation will make the compiler verify if the code really complies with the tail recursion characteristics.

In [7]:
import scala.annotation.tailrec

@tailrec
def sum3(n:Int,current:Int=0):Int =
  if (n == 1) then current + n
  else sum3(n-1,current + n)

sum3(50000)

import scala.annotation.tailrec


defined function sum3
res6_2: Int = 1250025000

Consider another function to get the length of a list, using a recursive call

In [ ]:
def computeLength(list: Seq[Int]): Int = list match
  case Nil => 0
  case first :: rest => 1 + computeLength(rest)

Do we have a tail or head recursion here?

In [10]:
val numbers = (1 to 20000).toList
computeLength(numbers)

numbers: List[Int] = List(
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
...
res9_1: Int = 20000

Again, if we make small modifications we can get a tail recursive call, and make the compiler check it with `@tailrec`.

In [9]:
@tailrec
def computeLength(list:Seq[Int], currentLength:Int=0):Int = list match
  case Nil => currentLength
  case first :: rest => computeLength(rest,currentLength + 1)

defined function computeLength

In [11]:
computeLength(numbers)

res10: Int = 20000

If we make anything after the recursive call, the compiler will complain.

In [11]:
@tailrec
def computeLength(list:Seq[Int], currentLength:Int=0):Int = list match
  case Nil => currentLength
  case first :: rest => 
    val value = computeLength(rest,currentLength + 1)
    value

-- Error: cmd11.sc:5:29 --------------------------------------------------------
5 |    val value = computeLength(rest,currentLength + 1)
  |                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |                Cannot rewrite recursive call: it is not in tail positionCompilation Failed

: 

In [ ]:
computeLength(numbers)

In certain situations, recursive calls are done indirectly, by calling another function, that ends up caling the original function again.

For example, consider this version of the `computeLength` fucntion:

In [12]:

def computeLength(list:Seq[Int], currentLength:Int=0):Int = list match
  case Nil => currentLength
  case first :: rest => computeLength2(rest,currentLength + 1)

def computeLength2(list:Seq[Int], currentLength:Int=0):Int = list match
  case Nil => currentLength
  case first :: rest => computeLength(rest,currentLength + 1)

defined function computeLength
defined function computeLength2

In [ ]:
computeLength(numbers)

The `TailCalls` utilities in Scala provide the `done` and `tailcall` constructs to address this problem.

In [14]:
import scala.util.control.TailCalls._

def computeLength(list:Seq[Int], currentLength:Int=0):TailRec[Int] = list match
  case Nil => done(currentLength)
  case first :: rest => tailcall(computeLength2(rest,currentLength + 1))

def computeLength2(list:Seq[Int], currentLength:Int=0):TailRec[Int] = list match
  case Nil => done(currentLength)
  case first :: rest => tailcall(computeLength(rest,currentLength + 1))

import scala.util.control.TailCalls._


defined function computeLength
defined function computeLength2

In [15]:
computeLength(numbers).result

res14: Int = 20000